# Reinforcement Learning From AI Feedback (RLAIF) with Serverless customization on SageMaker AI

## Lab 4.2 - Reward Model

In this notebook, we are going to prepare the reward model for later for fine-tuning Qwen 2.5 - 7B Instruct


---

## Prerequisites
### AWS Access Setup and dependencies

In [ ]:
import boto3
from sagemaker.core.helper.session_helper import Session, get_execution_role

sess = Session()
sagemaker_session_bucket = None

if sagemaker_session_bucket is None and sess is not None:
    # set to default bucket if a bucket name is not given
    sagemaker_session_bucket = sess.default_bucket()

try:
    role = get_execution_role()
except ValueError:
    iam = boto3.client("iam")
    role = iam.get_role(RoleName="sagemaker_execution_role")["Role"]["Arn"]

s3_client = boto3.client("s3")
sess = Session(default_bucket=sagemaker_session_bucket)
bucket_name = sess.default_bucket()

print(f"sagemaker role arn: {role}")
print(f"sagemaker bucket: {sess.default_bucket()}")
print(f"sagemaker session region: {sess.boto_region_name}")

project_prefix = "humanlike-rlaif"

---

## Prepare the reward prompt
For RLAIF fine-tuning, we need a **reward prompt** that instructs the evaluator model (the "judge") on how to score candidate responses during training. This prompt defines the criteria the judge uses to assign reward signals, which in turn guide the reinforcement learning optimization.

Our reward prompt instructs the judge to evaluate each model-generated response based on how natural and human-like it sounds, scoring from 0 (clearly artificial) to 1 (indistinguishable from human writing). The evaluation focuses on naturalness of language, tone consistency, and stylistic nuance — deliberately de-emphasizing completeness and length to avoid rewarding verbose but mechanical outputs.

### Inspect the reward prompt and upload it to S3


In [ ]:
reward_prompt = (
    "You are an expert human evaluator assessing how human-like and natural "
    "a text response sounds. Given the original question and an LLM-generated "
    "response, rate how human-like the response is based on the following aspects:\n\n"
    "- Naturalness and fluidity of language\n"
    "- Appropriateness and consistency of tone\n"
    "- Stylistic nuances and variability typical of human writing\n"
    "- Coherence and logical flow without artificial phrasing or repetitive structures\n"
    "- Avoidance of common LLM patterns (e.g., excessive hedging, bullet-point lists, "
    "overly formal or generic phrasing)\n\n"
    "Instructions:\n"
    "- Read the question carefully.\n"
    "- Evaluate whether the response sounds like it was written by a knowledgeable human "
    "in a natural conversation.\n"
    "- Ignore spelling errors.\n"
    "- Don't over-index on completeness and length of the answer.\n"
    "- Assign a score from 0 to 1 representing how human-like the model's reply sounds:\n"
    "   - 1 means indistinguishable from a natural human response.\n"
    "   - 0 means obviously artificial, unnatural, or inconsistent with human language norms.\n\n"
    "Output Format:\n"
    "Return a JSON object with EXACTLY the following structure (no extra text):\n"
    '{\n  "score": <numeric>,\n  "reasoning": "<brief explanation highlighting key '
    'differences or strengths>"\n}\n\n'
    "Prompt: {{ prompt }}\n"
    "LLM-generated response: {{ response }}"
)

# Save to file for use in the training job
reward_prompt_path = f"{project_prefix}_reward_prompt.txt"
with open(reward_prompt_path, "w") as f:
    f.write(reward_prompt)

In [ ]:
import boto3

s3_client = boto3.client('s3')
bucket_name = sess.default_bucket()

file_name = f"{project_prefix}_reward_prompt.txt"
prefix_key = f"{project_prefix}/{file_name}"
s3_client.upload_file(file_name, bucket_name, prefix_key)

reward_prompt_uri = f"s3://{bucket_name}/{prefix_key}"
print(reward_prompt_uri)

### Register SageMaker AI Evaluator with the Reward Prompt
We register the uploaded reward prompt as a reusable asset in the SageMaker AI Registry. This creates a versioned, addressable resource (identified by its ARN) that can be referenced by the RLAIF training job.

In [ ]:
from sagemaker.ai_registry.evaluator import Evaluator
from sagemaker.ai_registry.air_constants import REWARD_PROMPT

reward_prompt = Evaluator.create(
    name=f"{project_prefix}-reward-prompt",
    type=REWARD_PROMPT,
    source=reward_prompt_uri,
    wait=True)
    
print(f"Reward prompt ARN: {reward_prompt.arn}")
reward_prompt_arn=reward_prompt.arn